Question answering tasks return an answer given a question. If you've ever asked a virtual assistant like Alexa, Siri or Google what the weather is, then you've used a question answering model before. There are two common types of question answering tasks:

- Extractive: extract the answer from the given context.
- Abstractive: generate an answer from the context that correctly answers the question.

This guide will show you how to:

1. Finetune [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) on the [SQuAD](https://huggingface.co/datasets/squad) dataset for extractive question answering.
2. Use your finetuned model for inference.

In [ ]:
! pip install transformers datasets evaluate accelerate huggingface_hub

In [ ]:
user_name = "amin-oj"
from huggingface_hub import notebook_login
notebook_login()

## Load SQuAD dataset

In [ ]:
from datasets import load_dataset
squad = load_dataset("squad", split="train[:5000]")
squad = squad.train_test_split(test_size=0.2)
squad["train"][0]

There are several important fields here:

- `answers`: the starting location of the answer token and the answer text.
- `context`: background information from which the model needs to extract the answer.
- `question`: the question a model should answer.

## Preprocess

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

There are a few preprocessing steps particular to question answering tasks you should be aware of:

1. Some examples in a dataset may have a very long `context` that exceeds the maximum input length of the model. To deal with longer sequences, truncate only the `context` by setting `truncation="only_second"`.
2. Next, map the start and end positions of the answer to the original `context` by setting
   `return_offset_mapping=True`.
3. With the mapping in hand, now you can find the start and end tokens of the answer. Use the [sequence_ids](https://huggingface.co/docs/tokenizers/main/en/api/encoding#tokenizers.Encoding.sequence_ids) method to
   find which part of the offset corresponds to the `question` and which corresponds to the `context`.

Here is how you can create a function to truncate and map the start and end tokens of the `answer` to the `context`:

In [ ]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
tokenized_squad = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names
    # remove any columns you don't need
)

# a more efficeint preprocessing approach will be discussed later in the course
# it's essentially the same approach plus using overlapping contexts instead of
# truncaing them
# https://huggingface.co/learn/llm-course/chapter7/7

❓ what happens inside the pre_process function 👇

In [ ]:
# import pprint

# examples = squad['train'].shuffle(seed = 42).select(range(2)).to_dict()
# questions = [q.strip() for q in examples["question"]]

# # print(f"Questions: {questions}\n")
# # print(f"Contexts: {examples['context']}\n")

# inputs = tokenizer(
#     questions,
#     examples["context"],
#     max_length=384,
#     truncation="only_second",
#     return_offsets_mapping=True,
#     # Whether or not to return `(char_start, char_end)` for each token.
#     padding="max_length",
# )

# # pprint.pprint(f"tokenization output: {inputs}", indent = 2)

# offset_mapping = inputs.pop("offset_mapping")
# answers = examples["answers"]
# start_positions = []
# end_positions = []

# # a single iteration instead of a loop
# i = 0
# offset = offset_mapping[i]
# answer = answers[i]
# start_char = answer["answer_start"][0]
# end_char = answer["answer_start"][0] + len(answer["text"][0])
# sequence_ids = inputs.sequence_ids(i)
# # a list mapping the tokens to the id of their original sentences:
# # - `None` for special tokens added around or between sequences,
# # - `0` for tokens corresponding to words in the first sequence,
# # - `1` for tokens corresponding to words in the second sequence
# # when a pair of sequences was jointly encoded.


# # Find the start and end of the context
# idx = 0
# while sequence_ids[idx] != 1:
#     idx += 1
# context_start = idx
# while sequence_ids[idx] == 1:
#     idx += 1
# context_end = idx - 1

# # If the answer is not fully inside the context, label it (0, 0)
# if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
#     start_positions.append(0)
#     end_positions.append(0)
# else:
#     # Otherwise it's the start and end token positions
#     idx = context_start
#     while idx <= context_end and offset[idx][0] <= start_char:
#         idx += 1
#     start_positions.append(idx - 1)

#     idx = context_end
#     while idx >= context_start and offset[idx][1] >= end_char:
#         idx -= 1
#     end_positions.append(idx + 1)

# # print(f"context_start: {context_start}")
# # print(f"context_end: {context_end}")
# # print(f"start_positions: {start_positions}")
# # print(f"end_positions: {end_positions}")

Now create a batch of examples using [DefaultDataCollator](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DefaultDataCollator). Unlike other data collators in 🤗 Transformers, the [DefaultDataCollator](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DefaultDataCollator) does not apply any additional preprocessing such as padding.

In [ ]:
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

## Train

In [ ]:
from transformers import (AutoModelForQuestionAnswering,
                          TrainingArguments, Trainer)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

model_name = model_checkpoint.split("/")[-1]
task = "qa"
ckpt_name = f"{model_name}-finetuned-{task}"

training_args = TrainingArguments(
    output_dir=ckpt_name,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
trainer.push_to_hub()

## Evaluate

Evaluation for question answering requires a significant amount of postprocessing. To avoid taking up too much of your time, this guide skips the evaluation step. The [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) still calculates the evaluation loss during training so you're not completely in the dark about your model's performance.

If you have more time and you're interested in how to evaluate your model for question answering, take a look at the [Question answering](https://huggingface.co/course/chapter7/7?fw=pt#post-processing) chapter from the 🤗 Hugging Face Course!

## Inference

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline). Instantiate a `pipeline` for question answering with your model, and pass your text to it:

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model=f"{user_name}/{ckpt_name}")


question = "How many programming languages does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."


question_answerer(question=question, context=context)

You can also manually replicate the results of the `pipeline` if you'd like

In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering

# Tokenize the text and return PyTorch tensors
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(question, context, return_tensors="pt")

# Pass your inputs to the model and return the logits
model = AutoModelForQuestionAnswering\
.from_pretrained(f"{user_name}/{ckpt_name}")
with torch.no_grad():
    outputs = model(**inputs)


# Get the highest probability from the model output
# for the start and end positions
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

# Decode the predicted tokens to get the answer
predict_answer_tokens = inputs\
.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)